In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [4]:
df.head() # 34 Features, 1 category 

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [6]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [7]:
df.shape

(898, 35)

In [8]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [10]:
y.head()

0    BERHI
1    BERHI
2    BERHI
3    BERHI
4    BERHI
Name: Class, dtype: object

In [15]:
df["Class"].unique() #7 output layers

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [17]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [20]:
from  sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## ANN

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [27]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [28]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [30]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [34]:
#Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )
    def forward(self, x):
        return self.model(x)

In [35]:
model = ANN()

# loss and optimizer(optim)
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [36]:
# Training the NN
epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad() # initialize new value of the gradient each loop
        outputs = model(xb) # Forward Propagation
        loss = criteria(outputs, yb) # loss computation with gradient value
        loss.backward() # Backward Propagation
        optimizer.step() # parameters updation like weights

        running_loss += loss

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.6310254335403442
epoch = 2/100, loss = 1.0903100967407227
epoch = 3/100, loss = 0.7983099222183228
epoch = 4/100, loss = 0.6065860986709595
epoch = 5/100, loss = 0.4841163158416748
epoch = 6/100, loss = 0.41650518774986267
epoch = 7/100, loss = 0.3577387034893036
epoch = 8/100, loss = 0.3242940604686737
epoch = 9/100, loss = 0.29450350999832153
epoch = 10/100, loss = 0.273075670003891
epoch = 11/100, loss = 0.2545498311519623
epoch = 12/100, loss = 0.23908405005931854
epoch = 13/100, loss = 0.22575096786022186
epoch = 14/100, loss = 0.21066632866859436
epoch = 15/100, loss = 0.19846072793006897
epoch = 16/100, loss = 0.1868094801902771
epoch = 17/100, loss = 0.18186049163341522
epoch = 18/100, loss = 0.17111530900001526
epoch = 19/100, loss = 0.17126071453094482
epoch = 20/100, loss = 0.1578785926103592
epoch = 21/100, loss = 0.16470448672771454
epoch = 22/100, loss = 0.1539321094751358
epoch = 23/100, loss = 0.14599718153476715
epoch = 24/100, loss = 0.14074173

In [40]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # 7 output values will be produced
        _, predicted = torch.max(outputs, 1) # [0.1 0.2 0.9 1.5 -0.5 0.5 -0.9] so the max will print the max value index which 
                              # will be the correct class out off 0 to 6 total 7 classes as of now index value id 3.
        correct += (predicted == yb).sum().item() # y_test.shape will give the same value as this line
        total += yb.size(0) # actual samples in each batch

print("total values: ", total)
print("correct values: ", correct)
print("accuracy: ", correct/total * 100)

total values:  180
correct values:  168
accuracy:  93.33333333333333
